Nilufer Arsa Birim Degerleri - Wiki Push

Bronze katmandaki Nilufer arsa birim degerleri verisini okuyup Azure DevOps
Wiki'ye bir agent'in kolayca parse edebilecegi, yapilandirilmis (metadata +
schema JSON blogu + ornek veri tablosu) bir sayfa olarak yazar.

In [ ]:
%pip install tabulate

In [ ]:
%run "./Utils"

In [ ]:
import json
from datetime import datetime, timezone

DATALAKE_PATH = "abfss://axetproject@ozandatalake001.dfs.core.windows.net/axet_bronze/nilufer_arsa_birim_degerleri/"
WIKI_PAGE_PATH = "/Databricks/Nilufer_Arsa_Birim_Degerleri"

SOURCE_URL = "https://acikveri.nilufer.bel.tr/dataset/2026-arsa-birim-degerleri"
LICENSE = "CC BY 4.0 - Nilufer Belediyesi Acik Veri Lisansi"

df_bronze = read_from_datalake(DATALAKE_PATH, file_format="json")

In [ ]:
def get_datalake_size_bytes(path):

    total_size = 0

    for file_info in dbutils.fs.ls(path):
        if not file_info.isDir():
            total_size += file_info.size

    return total_size


def build_agent_friendly_content(df_spark, source_url, license_name, datalake_path):

    size_bytes = get_datalake_size_bytes(datalake_path)
    partition_count = df_spark.rdd.getNumPartitions()

    schema_fields = [
        {"name": field.name, "type": field.dataType.simpleString()}
        for field in df_spark.schema.fields
    ]

    metadata = {
        "dataset_name": "nilufer_arsa_birim_degerleri",
        "source_url": source_url,
        "license": license_name,
        "last_updated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "size_bytes": size_bytes,
        "partition_count": partition_count,
        "columns": schema_fields,
    }

    metadata_json = json.dumps(metadata, ensure_ascii=False, indent=2)

    schema_markdown = "\n".join(
        [
            f"| {field['name']} | {field['type']} |"
            for field in schema_fields
        ]
    )

    sample_json = df_spark.limit(10).toPandas().to_json(orient="records", force_ascii=False)
    sample_pretty = json.dumps(json.loads(sample_json), ensure_ascii=False, indent=2)

    content = f"""# Nilufer Arsa Birim Degerleri

## Metadata

```json
{metadata_json}
```

## Schema

| Column | Type |
|---|---|
{schema_markdown}

## Sample Data (first 10 rows, JSON)

```json
{sample_pretty}
```
"""

    return content

In [ ]:
content = build_agent_friendly_content(df_bronze, SOURCE_URL, LICENSE, DATALAKE_PATH)

response = push_wiki_page(
    WIKI_PAGE_PATH,
    content
)